In [2]:
import duckdb
import pandas as pd
import os
from datetime import datetime


In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

Depois de rodar com o arquivo = 'z0019_1.csv' é só substituir por arquivo = 'z0019_2.csv' e rodar novamente o notebook

In [12]:
arquivo = 'z0019_2.csv'
data_ingestão = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}' , sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestão
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2025-04-27 18:28:57.620361
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2025-04-27 18:28:57.620361
2,10003,PREGO,BT10,100,60,z0019_2.csv,2025-04-27 18:28:57.620361


Criar tabela para ingestao na camada bronze

In [13]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos (
         NATBR VARCHAR,
         MAKTX VARCHAR,
         WERKS VARCHAR,
         MAINS VARCHAR,   
         LABST VARCHAR,
         nome_arquivo VARCHAR,
         data_ingestao TIMESTAMP   
    )
""")

iNSERIR DOS DADOS NO BANCO DE DADOS TRAZENDO DO DATAFRAME

In [14]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")


In [19]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
print(resultado.head(6))

   NATBR     MAKTX WERKS MAINS LABST nome_arquivo              data_ingestao
0  10001  PARAFUSO  BT10   100   100  z0019_1.csv 2025-04-27 17:08:17.027682
1  10002   MARTELO  BT50   100  1500  z0019_1.csv 2025-04-27 17:08:17.027682
2  10003     PREGO  BT10   100    50  z0019_1.csv 2025-04-27 17:08:17.027682
3  10004     SERRA  BT50   100   200  z0019_2.csv 2025-04-27 18:28:57.620361
4  10005   MACHADO  BT50   100   100  z0019_2.csv 2025-04-27 18:28:57.620361
5  10003     PREGO  BT10   100    60  z0019_2.csv 2025-04-27 18:28:57.620361


Depois de inserir os dados na tabela, trazidos do arquivo z0019_1, agora só subistituir lá encima z0019_1 para trazer os dados da z0019_2 e vir executando o notebook novamente

Como estamos na camada bronze, é comum a camada trazer o nome do arquivo de origem, por isso vamos trocar de bronze_produtos para bronze_z0019.

In [17]:
con.execute("Alter Table bronze_produtos RENAME TO bronze_z0019")

Não esquecer de fechar a conecxão com o banco de dados.

In [21]:
con.close()